# Day 34 — Practice Questions
### More Than One Column

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A needs no computer** for most questions. Work it out, then run the cell to check.

Three questions to keep asking:

- What else is in the model? A coefficient means nothing without that list.
- Are two of my columns saying **the same thing**?
- Did I measure the gain on rows the model **never saw**?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("ready")

---
# Part A — Work it out

A delivery company fits this model to predict how many **minutes** a delivery takes:

```
minutes  =  2.4 × distance_km  +  1.8 × items  −  3.1 × rider_experience_years
            +  6.5 × is_dinner  +  11.0
```

Four input columns, one intercept.

### A1 · Predict one delivery

An order goes **5 km**, has **4 items**, the rider has **2 years** of experience, and it is
**dinner time** (`is_dinner = 1`).

How many minutes? Show each multiplication.

In [ ]:
def predict(distance_km, items, experience, is_dinner):
    return (2.4*distance_km + 1.8*items - 3.1*experience
            + 6.5*is_dinner + 11.0)

print(predict(5, 4, 2, 1))

### A2 · Say each coefficient out loud

Write one sentence for each of the four coefficients. Each sentence must contain the
clause that makes it true.

### A3 · What does the intercept mean here?

What would 11.0 describe, and should you quote it?

### A4 · Two deliveries

Delivery P: 8 km, 3 items, rider with 1 year, not dinner.
Delivery Q: 8 km, 3 items, rider with 5 years, not dinner.

How much faster is Q, and why can you answer without computing either one?

In [ ]:
print("P:", predict(8, 3, 1, 0))
print("Q:", predict(8, 3, 5, 0))
print("difference:", round(predict(8, 3, 5, 0) - predict(8, 3, 1, 0), 2))

### A5 · Which column matters most?

A colleague looks at the four coefficients — 2.4, 1.8, −3.1, 6.5 — and says
*"dinner time matters most, it has the biggest number."*

Is he right?

### A6 · A sign that flips

A model of exam marks is fitted twice:

| model | coefficient on `hours_of_sleep` |
|---|---|
| sleep alone | **+2.1** marks per hour |
| sleep **and** `hours_of_study` | **−0.4** marks per hour |

A student says one of the two must be wrong. Is she right? Explain what is happening.

### A7 · One-hot and the missing category

A model of flat rent includes a `city_area` column with four values: **Andheri**, **Powai**,
**Thane**, **Bandra**. After `pd.get_dummies(..., drop_first=True)` the model reports:

```
city_area_Bandra   +9,400
city_area_Powai    +2,100
city_area_Thane    −3,800
```

a. Which area is missing, and why?
b. What does `+9,400` actually mean?
c. How much more is a Bandra flat than a Powai flat, all else equal?

### A8 · Which model would you ship?

Three models of the same data, scored on rows they never saw:

| | columns | R² studied | R² hidden |
|---|---|---|---|
| **A** | 4 | 0.81 | 0.79 |
| **B** | 12 | 0.88 | 0.80 |
| **C** | 60 | 0.97 | 0.62 |

Which do you ship, and what is going on in each?

---
# Part B — Find the mistake

Four short programs. Each runs cleanly and reports something misleading.

In [ ]:
# ---------------------------------------------------------------
# 400 flats. B1 to B4 all use `flats`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3415)
n = 400

sqft     = rng.normal(950, 250, n).clip(400, 2000).round(0)
bedrooms = np.clip(np.round(sqft/440 + rng.normal(0, 0.7, n)), 1, 5)
age      = rng.uniform(0, 28, n).round(0)
rent     = (9500 + 22*sqft - 2800*bedrooms - 240*age + rng.normal(0, 3000, n)).round(0)

flats = pd.DataFrame({"sqft": sqft, "bedrooms": bedrooms, "age_years": age, "rent": rent})
flats.head()

### B1 · "Age barely matters"

In [ ]:
X = flats[["sqft", "bedrooms", "age_years"]]
y = flats["rent"]

model = LinearRegression().fit(X, y)

for name, value in zip(X.columns, model.coef_):
    print(f"{name:<12} {value:>+10.2f}")
print()
print("Conclusion: age_years has the smallest effect, so we can drop it.")   # <-- look here

### B2 · The helpful intern adds a column

In [ ]:
work = flats.copy()
work["sqm"] = (work.sqft * 0.0929).round(2)      # the same area, in square metres

cols = ["sqft", "sqm", "bedrooms", "age_years"]

for seed in [0, 1, 2]:
    a, b, c, d = train_test_split(work[cols], work.rent, test_size=0.3, random_state=seed)
    m = LinearRegression().fit(a, c)
    print(f"split {seed}:  sqft {m.coef_[0]:>+11.2f}   sqm {m.coef_[1]:>+12.2f}"
          f"   hidden R² {m.score(b, d):.4f}")

### B3 · "Every column we add makes it better"

In [ ]:
noise_rng = np.random.default_rng(5)
padded = flats.copy()
for j in range(25):
    padded[f"extra_{j}"] = noise_rng.normal(0, 1, n)     # pure random numbers

base_cols = ["sqft", "bedrooms", "age_years"]
all_cols = base_cols + [f"extra_{j}" for j in range(25)]

for cols_now in [base_cols, all_cols]:
    m = LinearRegression().fit(padded[cols_now], padded.rent)
    print(f"{len(cols_now):>2} columns  ->  R² {m.score(padded[cols_now], padded.rent):.4f}")
print()
print("Conclusion: more columns, better model.")    # <-- look here

### B4 · The report that goes to the landlord

In [ ]:
m = LinearRegression().fit(flats[["sqft", "bedrooms", "age_years"]], flats.rent)
print(f"coefficient on bedrooms: {m.coef_[1]:+.2f}")
print()
print(f"REPORT: 'Bedrooms reduce rent by ₹{abs(m.coef_[1]):,.0f} each. We recommend")
print( "         converting all 3BHK units into 1BHK to raise income.'")

---
# Part C — Write the code

Five tasks on one dataset.

In [ ]:
# ---------------------------------------------------------------
# 600 used cars. C1 to C5 all use `cars`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3410)
N = 600

age_years = rng.uniform(0, 14, N).round(1)
km_driven = (age_years*11000 + rng.normal(0, 18000, N)).clip(500, None).round(0)
engine_cc = rng.choice([800, 1000, 1200, 1500, 2000], N, p=[.15, .25, .3, .2, .1]).astype(float)
owners    = np.clip(np.round(1 + age_years/6 + rng.normal(0, 0.5, N)), 1, 4)
fuel      = rng.choice(["petrol", "diesel", "cng"], N, p=[.55, .35, .10])

bump = np.where(fuel == "diesel", 48000, np.where(fuel == "cng", -22000, 0))
price = (520000 - 31000*age_years - 0.42*km_driven + 150*engine_cc
         - 26000*owners + bump + rng.normal(0, 35000, N)).clip(40000, None).round(0)

cars = pd.DataFrame({"age_years": age_years, "km_driven": km_driven,
                     "engine_cc": engine_cc, "owners": owners,
                     "fuel": fuel, "price": price})
cars.head()

### C1 · Look first, including at the inputs

Do step 1 of the recipe properly:

- a scatter of each numeric column against `price`
- the correlation table **between the input columns**

Then write one sentence about what you see in the second one.

In [ ]:
# your code here

### C2 · Encode the word column, then fit

One-hot `fuel`, split, fit, and print every coefficient with its name. Report R² on both
the studied and the hidden rows, plus the average miss in rupees.

In [ ]:
# your code here

### C3 · Which column matters most?

Rank the six columns by actual influence. Show both the raw and the scaled coefficients
side by side, and say which column the raw numbers would have misled you about.

In [ ]:
# your code here

### C4 · Add the columns one at a time

Starting from `age_years` alone, add `km_driven`, then `engine_cc`, then `owners`, then
the fuel columns. Report the hidden R² and the average miss at each step.

Then answer: **did `km_driven` earn its place?**

In [ ]:
# your code here

### C5 · Price a specific car, and know when to refuse

Using your model from C2, price these three cars. For each one, decide whether you should
answer at all.

| | age | km | cc | owners | fuel |
|---|---|---|---|---|---|
| **A** | 6 years | 70,000 | 1,200 | 2 | petrol |
| **B** | 3 years | 160,000 | 1,500 | 1 | diesel |
| **C** | 25 years | 300,000 | 2,500 | 6 | petrol |

In [ ]:
# your code here

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Build multiple regression from scratch

Write **two** functions that fit a multi-column linear model, using no scikit-learn:

1. `fit_exact(X, y)` — the direct solution. Returns `(coefficients, intercept)`.
2. `fit_descent(X, y, rate, steps)` — gradient descent, generalised from Day 33.

Then prove both match `LinearRegression` on a dataset with **four** input columns.

**Marks are for:**
- both working for any number of columns, not just four
- `fit_descent` reporting its progress so a reader can see it converge
- handling the scale problem — `km_driven` is in the tens of thousands while `owners` is
  1 to 4, and plain gradient descent will struggle with that

*Hint for part 1: `np.linalg.lstsq` solves it, if you first add a column of ones to stand
in for the intercept.*

In [ ]:
demo = encoded[["age_years", "km_driven", "engine_cc", "owners"]].values
demo_y = cars.price.values
print(demo.shape, demo_y.shape)

In [ ]:
# your two functions here

### D2 · A model that blames the nurses

A hospital analytics team reports this to the board:

> *"We modelled patient deaths across 400 ward-shifts using beds, nurses on duty and
> equipment age. The model explains 60% of the variation on data it has never seen. The
> clearest finding is **nurses on duty**: each additional nurse on a shift is associated
> with **0.66 more deaths**. We recommend reviewing whether current staffing levels are
> excessive."*

The regression is correctly fitted. The split is correct. There is no bug in the code.

**Your tasks:**

1. Run it and confirm the numbers.
2. The dataset contains one column the team did **not** put in the model. Find it, and
   show what happens when you include it.
3. Prove *why* its absence produced that result — with evidence, not just assertion.
4. Write four or five sentences to send back to the board.

*Hints, in increasing order of giving it away: look at what columns exist · ask what
decides how many nurses a ward gets · compare wards that are similar in that respect.*

In [ ]:
rng = np.random.default_rng(5440)
n = 400

patient_severity = rng.uniform(1, 10, n).round(1)      # 1 = minor, 10 = critical
beds             = rng.integers(14, 41, n).astype(float)
nurses_on_duty   = (beds*0.16 + patient_severity*1.05
                    + rng.normal(0, 0.9, n)).clip(2, None).round(0)
equipment_age    = rng.uniform(0, 15, n).round(1)

deaths = (1.5 + 1.42*patient_severity - 0.55*nurses_on_duty
          + 0.09*beds + 0.06*equipment_age
          + rng.normal(0, 1.1, n)).clip(0, None).round(0)

wards = pd.DataFrame({"beds": beds, "nurses_on_duty": nurses_on_duty,
                      "equipment_age": equipment_age,
                      "patient_severity": patient_severity, "deaths": deaths})
wards.head()

In [ ]:
their_cols = ["beds", "nurses_on_duty", "equipment_age"]

a, b, c, d = train_test_split(wards[their_cols], wards.deaths, test_size=0.3, random_state=0)
their_model = LinearRegression().fit(a, c)

for name, value in zip(their_cols, their_model.coef_):
    print(f"   {name:<18} {value:>+9.4f}")
print(f"   {'intercept':<18} {their_model.intercept_:>+9.4f}")
print()
print(f"R² on hidden rows : {their_model.score(b, d):.4f}")

In [ ]:
# your investigation here

---
## Before you hand it in

- For every coefficient you quote: did you say **what else is in the model**?
- Did you **scale** before comparing coefficient sizes?
- Did you check the correlations **between the inputs**, not just with the answer?
- Did you measure every gain on rows the model **never saw**?
- For every prediction: is each value in range, **and did that combination ever occur**?
- Did you say "associated with" rather than "causes"?

Four numbers worth carrying away:

- Nurses: **+0.66** without severity, **−0.51** with it. The same column, the opposite
  advice, and one of them would have cut hospital staffing.
- Twenty-five junk columns raised training R² from **0.699 to 0.724** and lowered the
  honest score from **0.739 to 0.694**.
- `km_driven` has the smallest raw coefficient in the car model (**−0.52**) and the fourth
  largest influence. `fuel_diesel` has the largest raw coefficient and the third.
- `sqft` and `sqm` left the hidden R² **completely unchanged** and made the coefficients
  meaningless.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*